In [1]:
bronze_jobs_df = spark.table("bronze_jobs")

print("Bronze rows:", bronze_jobs_df.count())

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 3, Finished, Available, Finished, False)

Bronze rows: 20


In [2]:
bronze_jobs_df.show(5, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 4, Finished, Available, Finished, False)

+--------+-----------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
silver_jobs_df = bronze_jobs_df

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 5, Finished, Available, Finished, False)

In [4]:
for column in silver_jobs_df.columns:
    silver_jobs_df = silver_jobs_df.withColumnRenamed(
        column,
        column.strip().lower().replace(" ", "_")
    )

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 6, Finished, Available, Finished, False)

In [5]:
print(silver_jobs_df.columns)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 7, Finished, Available, Finished, False)

['job_id', 'job_title', 'description', 'company', 'publication_date', 'application_deadline', 'employment_type', 'working_hours', 'duration', 'workplace_model', 'city', 'region', 'country', 'occupation', 'occupation_field', 'job_url']


In [6]:
from pyspark.sql.functions import col, trim

text_columns = [
    "job_title",
    "company",
    "employment_type",
    "working_hours",
    "duration",
    "workplace_model",
    "city",
    "region",
    "country",
    "occupation",
    "occupation_field",
    "job_url"
]

for c in text_columns:
    silver_jobs_df = silver_jobs_df.withColumn(
        c,
        trim(col(c))
    )

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import when

for c in text_columns:
    silver_jobs_df = silver_jobs_df.withColumn(
        c,
        when(trim(col(c)) == "", None)
        .otherwise(trim(col(c)))
    )

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 9, Finished, Available, Finished, False)

In [8]:
silver_jobs_df.select(
    "publication_date",
    "application_deadline"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 10, Finished, Available, Finished, False)

+-------------------+--------------------+
|publication_date   |application_deadline|
+-------------------+--------------------+
|2026-03-31 17:50:38|2026-09-27 23:59:59 |
|2026-09-09 14:18:17|2026-09-16 23:59:59 |
|2026-09-04 15:38:03|2026-10-03 23:59:59 |
|2026-08-21 09:37:27|2026-09-20 23:59:59 |
|2026-09-11 15:40:58|2026-09-19 23:59:59 |
|2026-09-15 09:00:48|2026-09-16 23:59:59 |
|2026-09-15 16:25:07|2026-10-04 23:59:59 |
|2026-09-08 15:55:13|2026-10-08 23:59:59 |
|2026-09-09 10:34:34|2026-09-16 23:59:59 |
|2026-09-10 16:00:10|2026-10-05 23:59:59 |
|2026-09-15 09:43:44|2026-10-15 23:59:59 |
|2026-09-03 16:11:24|2026-10-30 23:59:59 |
|2026-08-21 15:12:21|2027-02-17 23:59:59 |
|2026-08-21 11:22:56|2027-02-17 23:59:59 |
|2026-08-18 09:30:22|2026-09-16 23:59:59 |
|2026-09-02 16:31:29|2026-10-02 23:59:59 |
|2026-04-02 11:57:42|2026-09-29 23:59:59 |
|2026-04-09 14:32:20|2026-09-30 23:59:59 |
|2026-09-15 14:35:12|2026-10-06 23:59:59 |
|2026-09-02 12:54:31|2027-03-01 23:59:59 |
+----------

In [9]:
from pyspark.sql.functions import to_timestamp, col

silver_jobs_df = silver_jobs_df.withColumn(
    "publication_date",
    to_timestamp(col("publication_date"), "yyyy-MM-dd HH:mm:ss")
)

silver_jobs_df = silver_jobs_df.withColumn(
    "application_deadline",
    to_timestamp(col("application_deadline"), "yyyy-MM-dd HH:mm:ss")
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 11, Finished, Available, Finished, False)

In [10]:
silver_jobs_df.printSchema()

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 12, Finished, Available, Finished, False)

root
 |-- job_id: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- company: string (nullable = true)
 |-- publication_date: timestamp (nullable = true)
 |-- application_deadline: timestamp (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- working_hours: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- workplace_model: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- occupation_field: string (nullable = true)
 |-- job_url: string (nullable = true)



In [11]:
silver_jobs_df.select(
    "job_id",
    "publication_date",
    "application_deadline"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 13, Finished, Available, Finished, False)

+--------+-------------------+--------------------+
|job_id  |publication_date   |application_deadline|
+--------+-------------------+--------------------+
|30842085|2026-03-31 17:50:38|2026-09-27 23:59:59 |
|31457113|2026-09-09 14:18:17|2026-09-16 23:59:59 |
|31440892|2026-09-04 15:38:03|2026-10-03 23:59:59 |
|31380520|2026-08-21 09:37:27|2026-09-20 23:59:59 |
|31468705|2026-09-11 15:40:58|2026-09-19 23:59:59 |
|31477008|2026-09-15 09:00:48|2026-09-16 23:59:59 |
|31481004|2026-09-15 16:25:07|2026-10-04 23:59:59 |
|31452883|2026-09-08 15:55:13|2026-10-08 23:59:59 |
|31455303|2026-09-09 10:34:34|2026-09-16 23:59:59 |
|31462986|2026-09-10 16:00:10|2026-10-05 23:59:59 |
|31477418|2026-09-15 09:43:44|2026-10-15 23:59:59 |
|31435751|2026-09-03 16:11:24|2026-10-30 23:59:59 |
|31383787|2026-08-21 15:12:21|2027-02-17 23:59:59 |
|31381643|2026-08-21 11:22:56|2027-02-17 23:59:59 |
|31363610|2026-08-18 09:30:22|2026-09-16 23:59:59 |
|31430761|2026-09-02 16:31:29|2026-10-02 23:59:59 |
|30859146|20

In [12]:
from pyspark.sql.functions import to_date, year, month

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 14, Finished, Available, Finished, False)

In [13]:
silver_jobs_df = silver_jobs_df.withColumn(
    "publication_date_only",
    to_date(col("publication_date"))
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 15, Finished, Available, Finished, False)

In [14]:
silver_jobs_df = silver_jobs_df.withColumn(
    "publication_year",
    year(col("publication_date"))
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 16, Finished, Available, Finished, False)

In [15]:
silver_jobs_df = silver_jobs_df.withColumn(
    "publication_month",
    month(col("publication_date"))
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 17, Finished, Available, Finished, False)

In [16]:
silver_jobs_df.select(
    "job_id",
    "publication_date",
    "publication_date_only",
    "publication_year",
    "publication_month"
).show(10, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 18, Finished, Available, Finished, False)

+--------+-------------------+---------------------+----------------+-----------------+
|job_id  |publication_date   |publication_date_only|publication_year|publication_month|
+--------+-------------------+---------------------+----------------+-----------------+
|30842085|2026-03-31 17:50:38|2026-03-31           |2026            |3                |
|31457113|2026-09-09 14:18:17|2026-09-09           |2026            |9                |
|31440892|2026-09-04 15:38:03|2026-09-04           |2026            |9                |
|31380520|2026-08-21 09:37:27|2026-08-21           |2026            |8                |
|31468705|2026-09-11 15:40:58|2026-09-11           |2026            |9                |
|31477008|2026-09-15 09:00:48|2026-09-15           |2026            |9                |
|31481004|2026-09-15 16:25:07|2026-09-15           |2026            |9                |
|31452883|2026-09-08 15:55:13|2026-09-08           |2026            |9                |
|31455303|2026-09-09 10:34:34|20

In [17]:
from pyspark.sql.functions import datediff

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 19, Finished, Available, Finished, False)

In [18]:
silver_jobs_df = silver_jobs_df.withColumn(
    "days_to_deadline",
    datediff(
        to_date(col("application_deadline")),
        to_date(col("publication_date"))
    )
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 20, Finished, Available, Finished, False)

In [19]:
silver_jobs_df.select(
    "job_id",
    "publication_date",
    "application_deadline",
    "days_to_deadline"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 21, Finished, Available, Finished, False)

+--------+-------------------+--------------------+----------------+
|job_id  |publication_date   |application_deadline|days_to_deadline|
+--------+-------------------+--------------------+----------------+
|30842085|2026-03-31 17:50:38|2026-09-27 23:59:59 |180             |
|31457113|2026-09-09 14:18:17|2026-09-16 23:59:59 |7               |
|31440892|2026-09-04 15:38:03|2026-10-03 23:59:59 |29              |
|31380520|2026-08-21 09:37:27|2026-09-20 23:59:59 |30              |
|31468705|2026-09-11 15:40:58|2026-09-19 23:59:59 |8               |
|31477008|2026-09-15 09:00:48|2026-09-16 23:59:59 |1               |
|31481004|2026-09-15 16:25:07|2026-10-04 23:59:59 |19              |
|31452883|2026-09-08 15:55:13|2026-10-08 23:59:59 |30              |
|31455303|2026-09-09 10:34:34|2026-09-16 23:59:59 |7               |
|31462986|2026-09-10 16:00:10|2026-10-05 23:59:59 |25              |
|31477418|2026-09-15 09:43:44|2026-10-15 23:59:59 |30              |
|31435751|2026-09-03 16:11:24|2026

In [20]:
silver_jobs_df.filter(
    col("application_deadline") < col("publication_date")
).select(
    "job_id",
    "publication_date",
    "application_deadline"
).show(truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 22, Finished, Available, Finished, False)

+------+----------------+--------------------+
|job_id|publication_date|application_deadline|
+------+----------------+--------------------+
+------+----------------+--------------------+



In [21]:
silver_jobs_df.filter(
    col("working_hours").isNull() |
    col("duration").isNull()
).select(
    "job_id",
    "job_title",
    "working_hours",
    "duration"
).show(truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 23, Finished, Available, Finished, False)

+--------+-----------------------------------------------------------+-------------+--------+
|job_id  |job_title                                                  |working_hours|duration|
+--------+-----------------------------------------------------------+-------------+--------+
|31440892|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI|NULL         |NULL    |
|31462986|Business Controller - Vikariat                             |NULL         |NULL    |
|31383787|IT-handläggare till PEAB i Ängelholm                       |NULL         |NULL    |
+--------+-----------------------------------------------------------+-------------+--------+



In [22]:
silver_jobs_df.select("employment_type").distinct().show(truncate=False)

silver_jobs_df.select("working_hours").distinct().show(truncate=False)

silver_jobs_df.select("workplace_model").distinct().show(truncate=False)

silver_jobs_df.select("duration").distinct().show(truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 24, Finished, Available, Finished, False)

+--------------------------------------------------------+
|employment_type                                         |
+--------------------------------------------------------+
|Vanlig anställning                                      |
|Tillsvidareanställning (inkl. eventuell provanställning)|
|Behovsanställning                                       |
+--------------------------------------------------------+

+-------------+
|working_hours|
+-------------+
|Heltid       |
|NULL         |
+-------------+

+---------------+
|workplace_model|
+---------------+
|Arbete på plats|
+---------------+

+----------------------+
|duration              |
+----------------------+
|Tills vidare          |
|6 månader eller längre|
|NULL                  |
+----------------------+



In [23]:
silver_jobs_df.select("job_id").show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 25, Finished, Available, Finished, False)

+--------+
|job_id  |
+--------+
|30842085|
|31457113|
|31440892|
|31380520|
|31468705|
|31477008|
|31481004|
|31452883|
|31455303|
|31462986|
|31477418|
|31435751|
|31383787|
|31381643|
|31363610|
|31430761|
|30859146|
|30881638|
|31479999|
|31428798|
+--------+



In [24]:
silver_jobs_df = silver_jobs_df.withColumn(
    "job_id",
    col("job_id").cast("long")
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 26, Finished, Available, Finished, False)

In [25]:
silver_jobs_df.printSchema()

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 27, Finished, Available, Finished, False)

root
 |-- job_id: long (nullable = true)
 |-- job_title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- company: string (nullable = true)
 |-- publication_date: timestamp (nullable = true)
 |-- application_deadline: timestamp (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- working_hours: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- workplace_model: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- occupation_field: string (nullable = true)
 |-- job_url: string (nullable = true)
 |-- publication_date_only: date (nullable = true)
 |-- publication_year: integer (nullable = true)
 |-- publication_month: integer (nullable = true)
 |-- days_to_deadline: integer (nullable = true)



In [26]:
silver_jobs_df.filter(
    col("job_id").isNull()
).select("job_id").show()

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 28, Finished, Available, Finished, False)

+------+
|job_id|
+------+
+------+



In [27]:
from pyspark.sql.functions import col, sum

silver_jobs_df.select(
    sum(col("job_id").isNull().cast("int")).alias("null_job_id"),
    sum(col("job_title").isNull().cast("int")).alias("null_job_title"),
    sum(col("company").isNull().cast("int")).alias("null_company"),
    sum(col("publication_date").isNull().cast("int")).alias("null_publication_date"),
    sum(col("application_deadline").isNull().cast("int")).alias("null_deadline"),
    sum(col("city").isNull().cast("int")).alias("null_city"),
    sum(col("description").isNull().cast("int")).alias("null_description")
).show()

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 29, Finished, Available, Finished, False)

+-----------+--------------+------------+---------------------+-------------+---------+----------------+
|null_job_id|null_job_title|null_company|null_publication_date|null_deadline|null_city|null_description|
+-----------+--------------+------------+---------------------+-------------+---------+----------------+
|          0|             0|           0|                    0|            0|        0|               0|
+-----------+--------------+------------+---------------------+-------------+---------+----------------+



In [28]:
from pyspark.sql.functions import lower

silver_jobs_df = silver_jobs_df.withColumn(
    "description_lower",
    lower(col("description"))
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 30, Finished, Available, Finished, False)

In [29]:
from pyspark.sql.functions import instr

silver_jobs_df = silver_jobs_df.withColumn(
    "has_power_bi",
    instr(col("description_lower"), "power bi") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 31, Finished, Available, Finished, False)

In [30]:
silver_jobs_df.select(
    "job_title",
    "has_power_bi"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 32, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+------------+
|job_title                                                              |has_power_bi|
+-----------------------------------------------------------------------+------------+
|Power BI Utvecklare                                                    |true        |
|Power BI Developer and Data Analyst                                    |true        |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |true        |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true        |
|Power Platform Specialist                                              |true        |
|Senior BI-utvecklare                                                   |true        |
|Controller med fokus på beslutsstöd och AI                             |true        |
|Service Owner Analytics & Senior BI Developer                          |true        |
|Business Controller                       

In [31]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_sql",
    instr(col("description_lower"), "sql") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 33, Finished, Available, Finished, False)

In [32]:
silver_jobs_df.select(
    "job_title",
    "has_sql"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 34, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+-------+
|job_title                                                              |has_sql|
+-----------------------------------------------------------------------+-------+
|Power BI Utvecklare                                                    |false  |
|Power BI Developer and Data Analyst                                    |true   |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false  |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true   |
|Power Platform Specialist                                              |false  |
|Senior BI-utvecklare                                                   |true   |
|Controller med fokus på beslutsstöd och AI                             |false  |
|Service Owner Analytics & Senior BI Developer                          |true   |
|Business Controller                                                    |false  |
|Business Contro

In [33]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_python",
    instr(col("description_lower"), "python") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 35, Finished, Available, Finished, False)

In [34]:
silver_jobs_df.select(
    "job_title",
    "has_python"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 36, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+----------+
|job_title                                                              |has_python|
+-----------------------------------------------------------------------+----------+
|Power BI Utvecklare                                                    |true      |
|Power BI Developer and Data Analyst                                    |false     |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false     |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true      |
|Power Platform Specialist                                              |false     |
|Senior BI-utvecklare                                                   |false     |
|Controller med fokus på beslutsstöd och AI                             |false     |
|Service Owner Analytics & Senior BI Developer                          |false     |
|Business Controller                                             

In [35]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_azure",
    instr(col("description_lower"), "azure") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 37, Finished, Available, Finished, False)

In [36]:
silver_jobs_df.select(
    "job_title",
    "has_azure"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 38, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+---------+
|job_title                                                              |has_azure|
+-----------------------------------------------------------------------+---------+
|Power BI Utvecklare                                                    |false    |
|Power BI Developer and Data Analyst                                    |true     |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false    |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true     |
|Power Platform Specialist                                              |true     |
|Senior BI-utvecklare                                                   |false    |
|Controller med fokus på beslutsstöd och AI                             |false    |
|Service Owner Analytics & Senior BI Developer                          |false    |
|Business Controller                                                    |fal

In [37]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_databricks",
    instr(col("description_lower"), "databricks") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 39, Finished, Available, Finished, False)

In [38]:
silver_jobs_df.select(
    "job_title",
    "has_databricks"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 40, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+--------------+
|job_title                                                              |has_databricks|
+-----------------------------------------------------------------------+--------------+
|Power BI Utvecklare                                                    |false         |
|Power BI Developer and Data Analyst                                    |false         |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false         |
|Junior Data Engineer | Azure | Databricks | Power BI                   |false         |
|Power Platform Specialist                                              |false         |
|Senior BI-utvecklare                                                   |true          |
|Controller med fokus på beslutsstöd och AI                             |false         |
|Service Owner Analytics & Senior BI Developer                          |true          |
|Business Controller 

In [39]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_fabric",
    instr(col("description_lower"), "fabric") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 41, Finished, Available, Finished, False)

In [40]:
silver_jobs_df.select(
    "job_title",
    "has_fabric"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 42, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+----------+
|job_title                                                              |has_fabric|
+-----------------------------------------------------------------------+----------+
|Power BI Utvecklare                                                    |false     |
|Power BI Developer and Data Analyst                                    |true      |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false     |
|Junior Data Engineer | Azure | Databricks | Power BI                   |false     |
|Power Platform Specialist                                              |false     |
|Senior BI-utvecklare                                                   |true      |
|Controller med fokus på beslutsstöd och AI                             |false     |
|Service Owner Analytics & Senior BI Developer                          |false     |
|Business Controller                                             

In [41]:
from pyspark.sql.functions import concat_ws

silver_jobs_df = silver_jobs_df.withColumn(
    "job_text",
    lower(
        concat_ws(
            " ",
            col("job_title"),
            col("description")
        )
    )
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 43, Finished, Available, Finished, False)

In [42]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_power_bi",
    instr(col("job_text"), "power bi") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 44, Finished, Available, Finished, False)

In [43]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_databricks",
    instr(col("job_text"), "databricks") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 45, Finished, Available, Finished, False)

In [44]:
silver_jobs_df.filter(
    col("job_title").contains("Junior Data Engineer")
).select(
    "job_title",
    "has_power_bi",
    "has_azure",
    "has_databricks",
    "has_fabric"
).show(truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 46, Finished, Available, Finished, False)

+----------------------------------------------------+------------+---------+--------------+----------+
|job_title                                           |has_power_bi|has_azure|has_databricks|has_fabric|
+----------------------------------------------------+------------+---------+--------------+----------+
|Junior Data Engineer | Azure | Databricks | Power BI|true        |true     |true          |false     |
+----------------------------------------------------+------------+---------+--------------+----------+



In [45]:
silver_jobs_df = silver_jobs_df.withColumn(
    "has_sql",
    instr(col("job_text"), "sql") > 0
)

silver_jobs_df = silver_jobs_df.withColumn(
    "has_python",
    instr(col("job_text"), "python") > 0
)

silver_jobs_df = silver_jobs_df.withColumn(
    "has_azure",
    instr(col("job_text"), "azure") > 0
)

silver_jobs_df = silver_jobs_df.withColumn(
    "has_fabric",
    instr(col("job_text"), "fabric") > 0
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 47, Finished, Available, Finished, False)

In [46]:
silver_jobs_df.select(
    "job_title",
    "has_power_bi",
    "has_sql",
    "has_python",
    "has_azure",
    "has_databricks",
    "has_fabric"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 48, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+------------+-------+----------+---------+--------------+----------+
|job_title                                                              |has_power_bi|has_sql|has_python|has_azure|has_databricks|has_fabric|
+-----------------------------------------------------------------------+------------+-------+----------+---------+--------------+----------+
|Power BI Utvecklare                                                    |true        |false  |true      |false    |false         |false     |
|Power BI Developer and Data Analyst                                    |true        |true   |false     |true     |false         |true      |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |true        |false  |false     |false    |false         |false     |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true        |true   |true      |true     |true          |false     |
|Power

In [47]:
from pyspark.sql.functions import regexp_extract

silver_jobs_df = silver_jobs_df.withColumn(
    "has_sql",
    regexp_extract(
        col("job_text"),
        r"\bsql\b",
        0
    ) != ""
)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 49, Finished, Available, Finished, False)

In [48]:
silver_jobs_df.select(
    "job_title",
    "has_sql"
).show(20, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 50, Finished, Available, Finished, False)

+-----------------------------------------------------------------------+-------+
|job_title                                                              |has_sql|
+-----------------------------------------------------------------------+-------+
|Power BI Utvecklare                                                    |false  |
|Power BI Developer and Data Analyst                                    |true   |
|Junior Prisanalytiker med erfarenhet av Mircrosoft Power BI            |false  |
|Junior Data Engineer | Azure | Databricks | Power BI                   |true   |
|Power Platform Specialist                                              |false  |
|Senior BI-utvecklare                                                   |true   |
|Controller med fokus på beslutsstöd och AI                             |false  |
|Service Owner Analytics & Senior BI Developer                          |true   |
|Business Controller                                                    |false  |
|Business Contro

In [49]:
from pyspark.sql.functions import sum, col

silver_jobs_df.select(
    sum(col("has_power_bi").cast("int")).alias("power_bi_jobs"),
    sum(col("has_sql").cast("int")).alias("sql_jobs"),
    sum(col("has_python").cast("int")).alias("python_jobs"),
    sum(col("has_azure").cast("int")).alias("azure_jobs"),
    sum(col("has_databricks").cast("int")).alias("databricks_jobs"),
    sum(col("has_fabric").cast("int")).alias("fabric_jobs")
).show()

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 51, Finished, Available, Finished, False)

+-------------+--------+-----------+----------+---------------+-----------+
|power_bi_jobs|sql_jobs|python_jobs|azure_jobs|databricks_jobs|fabric_jobs|
+-------------+--------+-----------+----------+---------------+-----------+
|           20|       8|          4|         5|              5|          3|
+-------------+--------+-----------+----------+---------------+-----------+



In [50]:
silver_jobs_df = silver_jobs_df.drop("description_lower")

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 52, Finished, Available, Finished, False)

In [51]:
silver_jobs_df.columns

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 53, Finished, Available, Finished, False)

['job_id',
 'job_title',
 'description',
 'company',
 'publication_date',
 'application_deadline',
 'employment_type',
 'working_hours',
 'duration',
 'workplace_model',
 'city',
 'region',
 'country',
 'occupation',
 'occupation_field',
 'job_url',
 'publication_date_only',
 'publication_year',
 'publication_month',
 'days_to_deadline',
 'has_power_bi',
 'has_sql',
 'has_python',
 'has_azure',
 'has_databricks',
 'has_fabric',
 'job_text']

In [52]:
print("Silver rows:", silver_jobs_df.count())
print("Unique job IDs:", silver_jobs_df.select("job_id").distinct().count())

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 54, Finished, Available, Finished, False)

Silver rows: 20
Unique job IDs: 20


In [53]:
silver_jobs_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_jobs")

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 55, Finished, Available, Finished, False)

In [54]:
silver_check_df = spark.table("silver_jobs")

print("Silver rows:", silver_check_df.count())
print("Unique job IDs:", silver_check_df.select("job_id").distinct().count())

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 56, Finished, Available, Finished, False)

Silver rows: 20
Unique job IDs: 20


In [55]:
silver_check_df.show(5, truncate=False)

StatementMeta(, 473d2d59-219b-4821-b715-c15d77144f83, 57, Finished, Available, Finished, False)

+--------+-----------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------